In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import re
from datetime import datetime
from groq import Groq
import time
from urllib.parse import urljoin, urlparse
import random

# Initialize Groq client
client = Groq(api_key="")

def llm_enhanced_scraping(target_url="https://dap-news.com/"):
    """
    Enhanced version that uses LLM for multiple scraping tasks.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    }
    
    print("🚀 STARTING LLM-ENHANCED SCRAPING")
    
    # Phase 1: LLM-assisted URL discovery
    print("\n📍 PHASE 1: LLM-Assisted URL Discovery")
    all_urls = llm_discover_urls(target_url, headers)
    
    if not all_urls:
        print("❌ No URLs found!")
        return []
    
    # Phase 2: LLM-enhanced content extraction
    print(f"\n📍 PHASE 2: LLM-Enhanced Content Extraction")
    all_articles = []
    
    for i, url in enumerate(all_urls):
        print(f"  🤖 [{i+1}/{len(all_urls)}] Processing with LLM: {url}")
        
        # Try LLM extraction first, fallback to traditional
        article_data = llm_extract_article(url, headers)
        if not article_data:
            article_data = scrape_single_article_fast(url, headers)
        
        if article_data:
            all_articles.append(article_data)
            print(f"    ✅ Success: {article_data['Title'][:60]}...")
        
        time.sleep(1)
    
    # Phase 3: LLM content analysis and enhancement
    print(f"\n📍 PHASE 3: LLM Content Analysis")
    all_articles = llm_analyze_articles(all_articles)
    
    return all_articles

def llm_discover_urls(base_url, headers):
    """Use LLM to intelligently discover article URLs."""
    discovered_urls = set()
    
    print("  🤖 Using LLM to analyze site structure...")
    
    try:
        response = requests.get(base_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get sample of page content for LLM analysis
        page_text = soup.get_text()[:2000]
        
        prompt = f"""
Analyze this Khmer news website content and suggest the best strategies to find article URLs:

Website: {base_url}
Content Sample: {page_text}

Based on this, what are the most likely URL patterns for news articles? 
What categories, pagination patterns, or URL structures should I look for?
Return only a JSON array of suggested URL patterns to try.

Example format:
{{
  "suggested_urls": [
    "https://dap-news.com/category/national/",
    "https://dap-news.com/page/2/",
    "https://dap-news.com/2024/10/22/sample-article/",
    "https://dap-news.com/news/12345/"
  ],
  "patterns": [
    "Look for /category/ pages",
    "Check pagination with /page/X/",
    "Articles often have dates in URLs"
  ]
}}
"""
        
        result = llm_query(prompt)
        if result:
            try:
                # Extract JSON from response
                json_match = re.search(r'\{.*\}', result, re.DOTALL)
                if json_match:
                    data = json.loads(json_match.group())
                    for url in data.get("suggested_urls", []):
                        discovered_urls.add(url)
                    print(f"    📊 LLM suggested {len(data.get('suggested_urls', []))} URL patterns")
            except:
                pass
        
    except Exception as e:
        print(f"  ⚠️ LLM URL discovery error: {e}")
    
    # Fallback to traditional discovery
    if not discovered_urls:
        print("  🔄 Falling back to traditional URL discovery...")
        traditional_urls = discover_all_urls_aggressive(base_url, headers)
        discovered_urls.update(traditional_urls)
    
    return list(discovered_urls)

def llm_extract_article(url, headers):
    """Use LLM to extract article content with better understanding."""
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get clean HTML content for LLM
        main_content = soup.find('article') or soup.find('main') or soup.find('body')
        content_sample = main_content.get_text()[:3000] if main_content else soup.get_text()[:3000]
        
        prompt = f"""
Extract the news article content from this Khmer text. Focus on:

1. ARTICLE TITLE (in Khmer)
2. MAIN CONTENT (in Khmer, complete paragraphs)
3. PUBLISH DATE (if available)
4. AUTHOR (if available)

Text content from {url}:
---
{content_sample}
---

Return as JSON:
{{
  "title": "full article title in Khmer",
  "content": "complete article content in Khmer", 
  "date": "YYYY-MM-DD or today's date if not found",
  "author": "author name or empty",
  "content_quality": "high/medium/low based on completeness"
}}
"""
        
        result = llm_query(prompt)
        if result:
            json_match = re.search(r'\{.*\}', result, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group())
                
                if data.get("title") and data.get("content"):
                    return {
                        "Title": data["title"],
                        "Content": data["content"],
                        "Date": data.get("date", datetime.now().strftime("%Y-%m-%d")),
                        "Source_URL": url,
                        "Author": data.get("author", ""),
                        "Content_Length": len(data["content"]),
                        "Content_Quality": data.get("content_quality", "unknown"),
                        "Extraction_Method": "LLM"
                    }
    
    except Exception as e:
        print(f"    ⚠️ LLM extraction failed: {e}")
    
    return None

def llm_analyze_articles(articles):
    """Use LLM to analyze and enhance the scraped articles."""
    enhanced_articles = []
    
    print("  🤖 Analyzing article quality and adding metadata...")
    
    for i, article in enumerate(articles):
        if i % 5 == 0:  # Analyze every 5th article to save tokens
            try:
                # Quality assessment
                quality_prompt = f"""
Assess this Khmer news article and provide metadata:

TITLE: {article['Title']}
CONTENT SAMPLE: {article['Content'][:800]}

Provide:
1. Topic/category (e.g., politics, sports, technology)
2. Key entities mentioned (people, organizations, locations)
3. Sentiment (positive/negative/neutral)
4. News value (high/medium/low)

Return as JSON:
{{
  "category": "main topic",
  "entities": ["entity1", "entity2"],
  "sentiment": "positive/negative/neutral", 
  "news_value": "high/medium/low",
  "summary_en": "brief 1-sentence English summary"
}}
"""
                analysis = llm_query(quality_prompt)
                if analysis:
                    json_match = re.search(r'\{.*\}', analysis, re.DOTALL)
                    if json_match:
                        data = json.loads(json_match.group())
                        
                        # Add LLM-generated metadata
                        article['Category'] = data.get("category", "")
                        article['Entities'] = ", ".join(data.get("entities", []))
                        article['Sentiment'] = data.get("sentiment", "")
                        article['News_Value'] = data.get("news_value", "")
                        
                        # Use LLM summary if available
                        if 'Summary_EN' not in article or not article['Summary_EN']:
                            article['Summary_EN'] = data.get("summary_en", "")
                
                print(f"    📊 Analyzed article {i+1}")
                
            except Exception as e:
                print(f"    ⚠️ Analysis failed for article {i+1}: {e}")
        
        enhanced_articles.append(article)
    
    return enhanced_articles

def llm_generate_dataset_report(articles):
    """Use LLM to generate a comprehensive report about the dataset."""
    if not articles:
        return
    
    print("\n📍 GENERATING DATASET REPORT WITH LLM")
    
    # Sample articles for analysis
    sample_titles = [article['Title'] for article in articles[:10]]
    sample_content = [article['Content'][:200] for article in articles[:5]]
    
    report_prompt = f"""
Analyze this Khmer news dataset and provide insights:

SAMPLE TITLES:
{sample_titles}

SAMPLE CONTENT:
{sample_content}

TOTAL ARTICLES: {len(articles)}

Please provide:
1. Main topics/themes covered
2. Date range and temporal patterns
3. Content quality assessment
4. Potential use cases for this data
5. Recommendations for improvement

Return as a structured report.
"""
    
    report = llm_query(report_prompt)
    if report:
        print("📊 LLM DATASET ANALYSIS REPORT:")
        print("=" * 50)
        print(report)
        print("=" * 50)
        
        # Save report to file
        with open(f"dataset_analysis_{datetime.now().strftime('%Y%m%d_%H%M')}.txt", "w", encoding="utf-8") as f:
            f.write(f"Dataset Analysis Report\n")
            f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Total Articles: {len(articles)}\n")
            f.write("=" * 50 + "\n")
            f.write(report)
        
        return report
    
    return None

# Update the main execution to use enhanced LLM features
if __name__ == "__main__":
    print("🌐 ENHANCED LLM-POWERED KHMER NEWS SCRAPER")
    print("=" * 50)
    
    # Use the LLM-enhanced scraper
    articles = llm_enhanced_scraping("https://dap-news.com/")
    
    if articles:
        # Generate LLM report
        llm_generate_dataset_report(articles)
        
        # Save dataset
        filename = f"llm_enhanced_news_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
        df = pd.DataFrame(articles)
        df.to_csv(filename, index=False, encoding='utf-8-sig')
        
        print(f"\n💾 Saved enhanced dataset: {filename}")
        print(f"📊 Enhanced features:")
        print(f"   - LLM-extracted content: {len(df[df['Extraction_Method'] == 'LLM'])} articles")
        print(f"   - Categories identified: {df['Category'].nunique()}")
        print(f"   - Content quality scores added")
        print(f"   - Entity extraction completed")

🌐 ENHANCED LLM-POWERED KHMER NEWS SCRAPER
🚀 STARTING LLM-ENHANCED SCRAPING

📍 PHASE 1: LLM-Assisted URL Discovery
  🤖 Using LLM to analyze site structure...
    📊 LLM suggested 28 URL patterns

📍 PHASE 2: LLM-Enhanced Content Extraction
  🤖 [1/28] Processing with LLM: https://dap-news.com/category/sport/
    ✅ Success: ពលករ និងនិស្សិតខ្មែរនៅកូរ៉េជាង ៤០០នាក់ ចូលរួមប្រកួតបាល់ទាត់ម...
  🤖 [2/28] Processing with LLM: https://dap-news.com/category/health/
    ✅ Success: ទារកម្នាក់កើតមកទម្ងន់ជិត៦គីឡូក្រាម កំពុងទទួលការព្យាបាល...
  🤖 [3/28] Processing with LLM: https://dap-news.com/category/covid-19/
    ✅ Success: អ្នកនិពន្ធបទចម្រៀងដ៏ពេញនិយម ទសវត្ស៩០ លក់ស្រែចូលបា កំពុងសង្គ្...
  🤖 [4/28] Processing with LLM: https://dap-news.com/category/national/page/2/
    ✅ Success: ជនបរទេសម្នាក់មកកម្ពុជា ដើម្បីបញ្ចប់ជីវិត តែបានផ្តល់ជីវិតថ្មី...
  🤖 [5/28] Processing with LLM: https://dap-news.com/category/economy/
    ✅ Success: អ្នកនិពន្ធបទចម្រៀងដ៏ពេញនិយម ទសវត្ស៩០ លក់ស្រែចូលបា កំពុងសង្គ្...
  🤖 [6/28] Pr

NameError: name 'scrape_single_article_fast' is not defined

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import re
from datetime import datetime
from groq import Groq
import time
from urllib.parse import urljoin, urlparse
import random

# Initialize Groq client
client = Groq(api_key="")

def llm_query(prompt, model="openai/gpt-oss-120b", max_retries=2):
    """LLM query with minimal retries for speed."""
    for attempt in range(max_retries):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model,
                temperature=0.1,
                max_tokens=200
            )
            return chat_completion.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                return ""
    return ""

def extract_khmer_text(text):
    """Fast Khmer text extraction."""
    if not text:
        return ""
    khmer_pattern = re.compile(r'[\u1780-\u17FF\u19E0-\u19FF\s\.\,\!\\?។៕៚៛]+')
    matches = khmer_pattern.findall(text)
    cleaned_text = ' '.join(matches).strip()
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    return cleaned_text

def is_valid_article_url(url, base_domain):
    """Check if URL is likely to be an article page."""
    parsed_url = urlparse(url)
    
    # Must be from same domain
    if base_domain not in parsed_url.netloc:
        return False
    
    # Common non-article patterns to exclude
    exclude_patterns = [
        r'sitemap', r'feed', r'rss', r'xml', r'wp-json',
        r'category/', r'tag/', r'author/', r'page/',
        r'search', r'comment', r'attachment', r'admin',
        r'\.pdf$', r'\.jpg$', r'\.png$', r'\.zip$',
        r'\?', r'#',  # Query strings and anchors
        r'\d{4}/\d{2}/\d{2}/?$'  # Date archive pages
    ]
    
    for pattern in exclude_patterns:
        if re.search(pattern, url, re.IGNORECASE):
            return False
    
    # Should have some path depth
    path_parts = [p for p in parsed_url.path.split('/') if p]
    if len(path_parts) < 2:
        return False
    
    # Should contain numeric ID or meaningful content
    has_numeric_id = any(re.search(r'\d+', part) for part in path_parts)
    has_content = len(path_parts[-1]) > 10 if path_parts else False
    
    return has_numeric_id or has_content

def get_smart_delay():
    """Minimal delay for maximum speed."""
    return random.uniform(0.5, 1.5)

def extract_article_content(soup, url):
    """Fast content extraction with multiple fallbacks."""
    
    # Priority selectors for content
    content_selectors = [
        '.td-post-content', '.tdb-block-inner', '.entry-content',
        '.post-content', '.article-content', '.content',
        '.story-content', 'article'
    ]
    
    for selector in content_selectors:
        content_elem = soup.select_one(selector)
        if content_elem:
            # Quick cleanup
            for unwanted in content_elem.select('.ads, .advertisement, script, style, iframe'):
                unwanted.decompose()
            
            content = extract_khmer_text(content_elem.get_text())
            if content and len(content) > 50:
                return content
    
    # Fallback: get all text and extract Khmer
    all_text = soup.get_text()
    khmer_text = extract_khmer_text(all_text)
    if len(khmer_text) > 100:
        return khmer_text
    
    return ""

def scrape_single_article_fast(url, headers):
    """Ultra-fast article scraper with minimal validation."""
    try:
        response = requests.get(url, headers=headers, timeout=8)
        
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Fast title extraction
        title = ""
        title_selectors = ['h1', '.entry-title', '.post-title', '.article-title', 'title']
        
        for selector in title_selectors:
            title_elem = soup.select_one(selector)
            if title_elem:
                title = extract_khmer_text(title_elem.get_text())
                if title:
                    break
        
        if not title:
            return None
        
        # Fast content extraction
        content = extract_article_content(soup, url)
        if not content or len(content) < 30:
            return None
        
        # Fast date extraction
        date_str = datetime.now().strftime("%Y-%m-%d")
        date_selectors = [
            'time.entry-date', 'time.post-date', '.published',
            'meta[property="article:published_time"]'
        ]
        
        for selector in date_selectors:
            date_elem = soup.select_one(selector)
            if date_elem:
                if date_elem.get('datetime'):
                    date_str = date_elem.get('datetime')[:10]
                break
        
        # Skip LLM summary for speed - we can add it later in batch
        article_data = {
            "Title": title,
            "Date": date_str,
            "Source_URL": url,
            "Content": content,
            "Content_Length": len(content),
            "Scraped_At": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        return article_data
        
    except Exception as e:
        return None

def discover_all_urls_aggressive(base_url, headers):
    """Aggressive URL discovery - finds ALL possible articles."""
    base_domain = urlparse(base_url).netloc
    discovered_urls = set()
    processed_pages = set()
    
    print("🔍 Starting aggressive URL discovery...")
    
    # Start with main pages
    queue = [
        base_url,
        f"{base_url.rstrip('/')}/page/1/",
        f"{base_url.rstrip('/')}/category/national/",
        f"{base_url.rstrip('/')}/category/politics/", 
        f"{base_url.rstrip('/')}/category/sport/",
        f"{base_url.rstrip('/')}/category/entertainment/",
        f"{base_url.rstrip('/')}/category/technology/",
    ]
    
    page_count = 0
    max_pages = 1000  # Safety limit to prevent infinite loops
    
    while queue and page_count < max_pages:
        current_url = queue.pop(0)
        
        if current_url in processed_pages:
            continue
            
        try:
            print(f"  📄 Scanning: {current_url}")
            response = requests.get(current_url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find ALL links on page
            links = soup.find_all('a', href=True)
            
            for link in links:
                href = link['href']
                full_url = urljoin(base_url, href)
                
                # Add valid article URLs
                if is_valid_article_url(full_url, base_domain):
                    discovered_urls.add(full_url)
                
                # Add new pages to queue for crawling
                elif (base_domain in urlparse(full_url).netloc and 
                      full_url not in processed_pages and
                      full_url not in queue and
                      not any(x in full_url for x in ['sitemap', 'feed', 'xml'])):
                    queue.append(full_url)
            
            # Look for pagination
            pagination_links = soup.find_all('a', href=True, 
                                           string=re.compile(r'[0-9]|next|older|newer|»|›'))
            for pagination_link in pagination_links:
                pagination_url = urljoin(base_url, pagination_link['href'])
                if (base_domain in urlparse(pagination_url).netloc and 
                    pagination_url not in processed_pages and
                    pagination_url not in queue):
                    queue.append(pagination_url)
            
            processed_pages.add(current_url)
            page_count += 1
            
            print(f"    ✅ Found {len(discovered_urls)} articles so far...")
            
            time.sleep(get_smart_delay())
            
        except Exception as e:
            print(f"  ⚠️ Error scanning {current_url}: {e}")
            continue
    
    url_list = list(discovered_urls)
    print(f"🎯 Discovery complete! Found {len(url_list)} potential articles")
    return url_list

def scrape_unlimited(target_url="https://dap-news.com/"):
    """
    UNLIMITED scraper - gets EVERYTHING it can find.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
    }
    
    print("🚀 STARTING UNLIMITED SCRAPING - NO LIMITS!")
    print("⚠️ This may take a while...")
    
    # Phase 1: Discover ALL URLs
    print("\n📍 PHASE 1: URL Discovery")
    all_urls = discover_all_urls_aggressive(target_url, headers)
    
    if not all_urls:
        print("❌ No URLs found!")
        return []
    
    # Phase 2: Scrape ALL articles
    print(f"\n📍 PHASE 2: Scraping {len(all_urls)} Articles")
    all_articles = []
    success_count = 0
    fail_count = 0
    
    for i, url in enumerate(all_urls):
        print(f"  [{i+1}/{len(all_urls)}] Processing: {url[:80]}...")
        
        article_data = scrape_single_article_fast(url, headers)
        if article_data:
            all_articles.append(article_data)
            success_count += 1
            print(f"    ✅ SUCCESS: {article_data['Title'][:60]}...")
        else:
            fail_count += 1
            print(f"    ❌ FAILED")
        
        # Progress update every 50 articles
        if (i + 1) % 50 == 0:
            success_rate = (success_count / (i + 1)) * 100
            print(f"📊 Progress: {i+1}/{len(all_urls)} | Success: {success_count} | Failed: {fail_count} | Rate: {success_rate:.1f}%")
        
        time.sleep(get_smart_delay())
    
    # Phase 3: Add summaries in batch (optional)
    if all_articles:
        print(f"\n📍 PHASE 3: Adding Summaries (Optional)")
        for i, article in enumerate(all_articles):
            if i % 10 == 0:  # Add summary to every 10th article to save time
                content_preview = article['Content'][:500]
                summary = llm_query(f"Briefly summarize in English: {content_preview}")
                article['Summary_EN'] = summary
                print(f"  📝 Added summary for article {i+1}")
            else:
                article['Summary_EN'] = ""
    
    print(f"\n🎉 UNLIMITED SCRAPING COMPLETE!")
    print(f"📊 FINAL RESULTS:")
    print(f"   Total URLs found: {len(all_urls)}")
    print(f"   Successfully scraped: {success_count}")
    print(f"   Failed: {fail_count}")
    print(f"   Success rate: {(success_count/len(all_urls))*100:.1f}%")
    
    return all_articles

def save_large_dataset(data, filename=None):
    """Save large datasets with compression options."""
    if not data:
        print("⚠️ No data to save.")
        return None
    
    if not filename:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"khmer_news_COMPLETE_{timestamp}.csv"
    
    df = pd.DataFrame(data)
    
    # Basic cleanup
    df = df.drop_duplicates(subset=['Source_URL'], keep='first')
    df = df[df['Content_Length'] > 50]  # Remove very short articles
    
    # Save to CSV
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    
    print(f"💾 SAVED COMPLETE DATASET: {filename}")
    print(f"📁 Final dataset size: {len(df)} articles")
    print(f"📊 Content statistics:")
    print(f"   - Average content length: {df['Content_Length'].mean():.0f} chars")
    print(f"   - Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"   - Total characters: {df['Content_Length'].sum():,}")
    
    return filename

def continuous_scraping(target_url="https://dap-news.com/", hours=24):
    """
    Continuous scraping mode - runs for specified hours.
    """
    print(f"🔄 STARTING CONTINUOUS SCRAPING FOR {hours} HOURS")
    print("Press Ctrl+C to stop early")
    
    start_time = time.time()
    end_time = start_time + (hours * 3600)
    all_articles = []
    cycle = 1
    
    try:
        while time.time() < end_time:
            print(f"\n🔄 CYCLE {cycle} - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            
            # Scrape everything
            new_articles = scrape_unlimited(target_url)
            all_articles.extend(new_articles)
            
            # Remove duplicates
            unique_articles = []
            seen_urls = set()
            for article in all_articles:
                if article['Source_URL'] not in seen_urls:
                    unique_articles.append(article)
                    seen_urls.add(article['Source_URL'])
            
            all_articles = unique_articles
            
            # Save progress
            if all_articles:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                progress_file = f"khmer_news_PROGRESS_cycle{cycle}_{timestamp}.csv"
                save_large_dataset(all_articles, progress_file)
            
            print(f"📈 Total unique articles so far: {len(all_articles)}")
            
            # Wait before next cycle (1 hour)
            next_cycle_time = time.time() + 3600
            while time.time() < next_cycle_time and time.time() < end_time:
                remaining = min(next_cycle_time - time.time(), end_time - time.time())
                if remaining > 0:
                    print(f"⏳ Next cycle in {remaining/60:.1f} minutes...")
                    time.sleep(300)  # Check every 5 minutes
            
            cycle += 1
    
    except KeyboardInterrupt:
        print("\n🛑 Continuous scraping stopped by user")
    
    print(f"\n🎉 CONTINUOUS SCRAPING FINISHED!")
    print(f"🕒 Total duration: {(time.time() - start_time)/3600:.1f} hours")
    print(f"📚 Total unique articles collected: {len(all_articles)}")
    
    # Save final dataset
    if all_articles:
        final_file = save_large_dataset(all_articles, "khmer_news_FINAL_COMPLETE.csv")
        return final_file
    
    return None

# Main execution - CHOOSE YOUR MODE:
if __name__ == "__main__":
    print("🌐 KHMER NEWS UNLIMITED SCRAPER")
    print("=" * 50)
    
    # MODE 1: One-time unlimited scrape
    print("MODE 1: One-time unlimited scraping")
    articles = scrape_unlimited("https://dap-news.com/")
    
    if articles:
        csv_path = save_large_dataset(articles)
        
        # Show sample
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
        print(f"\n📋 SAMPLE OF {len(df)} ARTICLES:")
        for i, (_, row) in enumerate(df.head(10).iterrows()):
            print(f"  {i+1}. {row['Title'][:80]}...")
    
    # UNCOMMENT FOR MODE 2: Continuous scraping (24 hours)
    """
    print("\n" + "="*50)
    print("MODE 2: Continuous scraping (24 hours)")
    continuous_scraping("https://dap-news.com/", hours=24)
    """

🌐 KHMER NEWS UNLIMITED SCRAPER
MODE 1: One-time unlimited scraping
🚀 STARTING UNLIMITED SCRAPING - NO LIMITS!
⚠️ This may take a while...

📍 PHASE 1: URL Discovery
🔍 Starting aggressive URL discovery...
  📄 Scanning: https://dap-news.com/
    ✅ Found 55 articles so far...
  📄 Scanning: https://dap-news.com/page/1/
    ✅ Found 55 articles so far...
  📄 Scanning: https://dap-news.com/category/national/
    ✅ Found 65 articles so far...
  📄 Scanning: https://dap-news.com/category/politics/
    ✅ Found 65 articles so far...
  📄 Scanning: https://dap-news.com/category/sport/
    ✅ Found 75 articles so far...
  📄 Scanning: https://dap-news.com/category/entertainment/
    ✅ Found 75 articles so far...
  📄 Scanning: https://dap-news.com/category/technology/
    ✅ Found 85 articles so far...
  📄 Scanning: https://dap-news.com/category/dailyevent/
    ✅ Found 95 articles so far...
  📄 Scanning: https://dap-news.com/latest-news/
    ✅ Found 102 articles so far...
  📄 Scanning: https://dap-news.co